# Synthetic User - pipeline walkthrough (observe & rectify)

Runs the whole single-turn pipeline step by step and shows, for **every node**:

1. the **inputs** it receives,
2. the exact **prompts** sent to the LLM (system + user, including retry feedback),
3. the **raw LLM output** (before any parsing),
4. what the **code** did with it (validation, integrity checks, downgrades, rejections + reasons),
5. the final **output** that flows into the next node.

```
Catalog -> [1] Reference Index -> [2] Scenario + Cheat Sheet -> [3] Question
        -> [4] Product Executor (STAND-IN) -> [5] SQL Evaluator -> Bug Candidates
```

**How to use:** pick a provider in the *Settings* cell, then `Run All`. Collapsed blocks
(&#9656;) hold prompts and raw outputs - click to expand. Every artifact (including all LLM
calls) is saved under `artifacts/notebook_runs/<run_id>/`.

> Node 4 (product executor) is not built yet. Section 4 uses a clearly-labelled **stand-in**
> NL2SQL product (the same LLM writing SQL) so the evaluator has something to judge. You can
> override any SQL by hand in that section. Generated SQL is **never executed**.

## Settings

In [1]:
# ---- knobs you will want to change -------------------------------------------------------
PROVIDER = "gemini"          # "gemini" = real LLM (needs GEMINI_API_KEY/GOOGLE_API_KEY + GEMINI_MODEL in .env)
                             # "fake"   = offline scripted responses (deterministic demo)
SCENARIO_COUNT = 2           # scenarios to request from Node 2
QUESTIONS_PER_SCENARIO = 2   # questions per scenario from Node 3
LLM_TIMEOUT_SECONDS = 120    # Gemini can be slow under load
SHOW_FULL_PROMPTS = True     # False = hide system/user prompts, show only raw outputs

In [2]:
import json
import os
import time
from dataclasses import asdict, dataclass
from datetime import UTC, datetime
from html import escape
from pathlib import Path
from typing import Any

from IPython.display import HTML, Markdown, display
from pydantic import BaseModel, TypeAdapter

REPO = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
os.chdir(REPO)

from dotenv import load_dotenv

load_dotenv(REPO / ".env", override=False)

from synthetic_user.core.config import LLMProvider, load_config
from synthetic_user.core.exceptions import LLMOutputValidationError, LLMProviderError
from synthetic_user.core.identifiers import SequentialIdGenerator, new_run_id
from synthetic_user.core.loaders import load_catalog, load_persona, load_product
from synthetic_user.core.logging import get_run_logger, setup_logging
from synthetic_user.llm.factory import create_llm_client
from synthetic_user.llm.fake import FakeLLMClient
from synthetic_user.llm.json_utils import extract_json_object
from synthetic_user.models.evaluation import BugCandidate, EvaluationResult
from synthetic_user.models.execution import EXECUTION_RESULT_ADAPTER, ExecutionResult
from synthetic_user.models.question import QuestionCase, RejectedQuestion
from synthetic_user.models.reference import ReferenceSnapshot
from synthetic_user.models.scenario import RejectedScenario, Scenario
from synthetic_user.nodes.question_generator import QuestionGenerator
from synthetic_user.nodes.reference_index import ReferenceIndexGenerator
from synthetic_user.nodes.scenario_generator import ScenarioGenerator
from synthetic_user.nodes.sql_evaluator import EvaluationCase, SqlEvaluator
from synthetic_user.nodes.sql_rules import apply_sql_rules
from synthetic_user.references.integrity import check_index_against_catalog
from synthetic_user.references.schema_leak import catalog_identifiers
from synthetic_user.storage.json_store import JsonArtifactStore

setup_logging("WARNING")  # warnings (e.g. retries) still show below cells
RUN_ID = new_run_id()
log = get_run_logger("notebook", RUN_ID)

settings = load_config(REPO / "config" / "app.yaml")
llm_settings = settings.llm.model_copy(
    update={
        "provider": LLMProvider(PROVIDER),
        "model": os.environ.get("GEMINI_MODEL", "").strip() or settings.llm.model
        if PROVIDER == "gemini"
        else "fake",
        "timeout_seconds": LLM_TIMEOUT_SECONDS,
    }
)
settings = settings.model_copy(update={"llm": llm_settings})

NB_DIR = REPO / "artifacts" / "notebook_runs" / RUN_ID
store = JsonArtifactStore(NB_DIR)
ids = SequentialIdGenerator()
print(f"run_id={RUN_ID}  provider={settings.llm.provider}  model={settings.llm.model}")
print(f"artifacts -> {NB_DIR}")

run_id=RUN_20261014_102436_05b4  provider=gemini  model=gemini-3.5-flash-lite
artifacts -> C:\Users\dhruv\synthetic_user_code\artifacts\notebook_runs\RUN_20261014_102436_05b4


### Display helpers and the recording LLM wrapper

In [3]:
def md(text: str) -> None:
    display(Markdown(text))


def _plain(obj: Any) -> Any:
    if isinstance(obj, BaseModel):
        return obj.model_dump(mode="json")
    if isinstance(obj, list):
        return [_plain(o) for o in obj]
    return obj


def show_json(obj: Any, title: str = "JSON", open_: bool = False) -> None:
    text = json.dumps(_plain(obj), indent=2, ensure_ascii=False, default=str)
    show_text(title, text, open_)


def show_text(title: str, text: str, open_: bool = False) -> None:
    flag = " open" if open_ else ""
    display(HTML(
        f"<details{flag}><summary><b>{escape(title)}</b> ({len(text):,} chars)</summary>"
        f"<pre style='white-space:pre-wrap;max-height:500px;overflow:auto'>{escape(text)}</pre>"
        "</details>"
    ))


def _cell(value: Any) -> str:
    if isinstance(value, (list, tuple, set)):
        value = ", ".join(str(v) for v in value) if value else "-"
    text = "-" if value is None or value == "" else str(value)
    return text.replace("|", "\\|").replace("\n", " ")


def table(rows: list[dict[str, Any]], columns: list[str] | None = None) -> None:
    if not rows:
        md("_(none)_")
        return
    columns = columns or list(rows[0])
    lines = ["| " + " | ".join(columns) + " |", "|" + "---|" * len(columns)]
    lines += ["| " + " | ".join(_cell(r.get(c)) for c in columns) + " |" for r in rows]
    md("\n".join(lines))


@dataclass
class LLMCall:
    stage: str
    method: str
    system: str
    user: str
    raw: str | None
    error: str | None
    seconds: float


class RecordingLLM:
    """Wraps the real client and records every call (prompts + raw text output).

    generate_json goes through generate_text + extract_json_object, which is exactly what
    the Gemini adapter does, so the raw text you see is what the parser saw.
    """

    def __init__(self, inner: Any) -> None:
        self.inner = inner
        self.stage = "-"
        self.calls: list[LLMCall] = []

    @property
    def provider_name(self) -> str:
        return self.inner.provider_name

    @property
    def model_name(self) -> str:
        return self.inner.model_name

    async def generate_text(self, system: str, user: str) -> str:
        start = time.perf_counter()
        try:
            text = await self.inner.generate_text(system, user)
        except Exception as exc:
            self.calls.append(LLMCall(self.stage, "text", system, user, None,
                                      f"{type(exc).__name__}: {exc}", time.perf_counter() - start))
            raise
        self.calls.append(LLMCall(self.stage, "text", system, user, text, None,
                                  time.perf_counter() - start))
        return text

    async def generate_json(self, system: str, user: str) -> dict[str, Any]:
        text = await self.generate_text(system, user)
        self.calls[-1].method = "json"
        try:
            return extract_json_object(text)
        except LLMOutputValidationError as exc:
            self.calls[-1].error = "; ".join(exc.errors)
            raise

    def show(self, stage: str) -> None:
        calls = [c for c in self.calls if c.stage == stage]
        md(f"**{len(calls)} LLM call(s)** in `{stage}`, total {sum(c.seconds for c in calls):.1f}s")
        for i, c in enumerate(calls, start=1):
            retry = " - RETRY (previous output was rejected)" if "previous output invalid:" in c.user else ""
            md(f"#### Call {i} ({c.method}, {c.seconds:.1f}s){retry}")
            if SHOW_FULL_PROMPTS:
                show_text("System prompt", c.system)
                show_text("User prompt", c.user)
                if retry:
                    show_text("Retry feedback the code sent", c.user.split("previous output invalid:", 1)[1], open_=True)
            show_text("Raw LLM output", c.raw or "(no output)", open_=True)
            if c.error:
                md(f"> **Error on this call:** `{c.error}`")


import asyncio

NODE_ATTEMPTS = 3  # re-run a node if the provider is overloaded (e.g. Gemini 503)


async def run_node(make_coro, attempts: int = NODE_ATTEMPTS):
    """Notebook-only convenience: retry a whole node on LLMProviderError with a short pause."""
    for attempt in range(1, attempts + 1):
        try:
            return await make_coro()
        except LLMProviderError as exc:
            if attempt == attempts:
                raise
            wait = 15 * attempt
            print(f"provider error (attempt {attempt}/{attempts}): {str(exc)[:160]} - retrying in {wait}s")
            await asyncio.sleep(wait)


md("helpers ready")

helpers ready

### LLM client

`gemini` uses the real adapter. `fake` uses scripted responses (from the test fixtures) so the
notebook runs offline; the fake data deliberately contains a few mistakes (an invented table,
a schema-leaking question, a wrong join) so you can watch the guards work.

In [4]:
FIX = REPO / "tests" / "fixtures"


def _fake_router(system: str, user: str) -> Any:
    stage = rec.stage
    if stage == "node1":
        index = json.loads((FIX / "sample_reference_index.json").read_text(encoding="utf-8"))
        return {"tables": index["tables"], "uncertainties": index["uncertainties"]}
    if stage == "node2":
        drafts = []
        for s in json.loads((FIX / "benchmark" / "scenarios.json").read_text(encoding="utf-8")):
            for key in ("id", "persona_id", "product_id", "reference_index_id", "signature"):
                s.pop(key)
            drafts.append(s)
        bad = json.loads(json.dumps(drafts[1]))
        bad["title"] = "Budget vs actual (invents a table)"
        bad["cheat_sheet"]["tables"].append("budgets")
        bad["cheat_sheet"]["columns"].append("budgets.limit_amount")
        return {"scenarios": [*drafts, bad]}
    if stage == "node3":
        def q(text: str, style: str = "precise", amb: str = "low") -> dict[str, Any]:
            return {"utterance": text, "intent": {"task_type": "ranking"},
                    "question_characteristics": {"style": style, "ambiguity_level": amb},
                    "expectations": {"clarification": "conditional",
                                     "acceptable_behaviors": ["handles currencies explicitly"],
                                     "unacceptable_behaviors": ["sums across currencies silently"]},
                    "interpretation_notes": ["'spend' may need a currency choice"]}
        if "Top spenders by employee" in user:
            return {"questions": [q("Who were our top 10 card spenders over the last three months?"),
                                  q("Which staff are spending the most on their cards lately?", "investigative", "medium")]}
        return {"questions": [q("How has monthly card spend per merchant category moved this year?"),
                              q("Show monthly SUM of amount from card_transactions per merchant_category")]}
    if stage == "node4":
        if "top 10 card spenders" in user:
            return ("SELECT e.full_name, c.currency_code, SUM(c.amount) AS total_spend\n"
                    "FROM card_transactions AS c JOIN employees AS e ON c.employee_id = e.employee_id\n"
                    "WHERE c.transaction_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 3 MONTH)\n"
                    "GROUP BY e.full_name, c.currency_code ORDER BY total_spend DESC LIMIT 10")
        if "spending the most" in user:
            return ("SELECT e.full_name, SUM(c.amount) AS total_spend\n"
                    "FROM card_transactions AS c JOIN employees AS e ON c.merchant_id = e.employee_id\n"
                    "GROUP BY e.full_name ORDER BY total_spend DESC")
        if "merchant category" in user:
            return "CLARIFY: Should amounts be shown per currency or converted to one currency?"
        return "SELEC merchant_category FROM card_transactions"
    if stage == "node5":
        if "clarification" in system:
            return {"relevance": "relevant", "necessity": "necessary", "asks_for_given_info": False,
                    "reason": "Amounts are in several currencies and the question does not say which."}
        checks = dict.fromkeys(["intent_alignment", "table_selection", "column_selection",
                                "join_correctness", "filter_correctness", "aggregation_correctness",
                                "sql_syntax", "semantic_considerations"], "pass")
        return {"checks": checks, "findings": [], "cheat_sheet_conflicts": [], "verdict": "pass"}
    raise AssertionError(f"no fake response for stage {stage}")


if PROVIDER == "fake":
    inner = FakeLLMClient([_fake_router] * 1000, model_name="fake")
else:
    inner = create_llm_client(settings)
rec = RecordingLLM(inner)
print(f"LLM client: {rec.provider_name} / {rec.model_name}")

LLM client: gemini / gemini-3.5-flash-lite


---
## 0. Inputs (persona, product, catalog)

In [5]:
persona = load_persona(settings.inputs.persona_path)
product = load_product(settings.inputs.product_path)
catalog = load_catalog(settings.inputs.catalog_path)

md(f"### Persona `{persona.id}` v{persona.version}")
table([{"field": "role", "value": persona.identity.role},
       {"field": "expertise (domain / sql)", "value": f"{persona.expertise.domain} / {persona.expertise.sql}"},
       {"field": "style", "value": persona.behavior.style},
       {"field": "traits", "value": persona.behavior.traits},
       {"field": "motivation", "value": persona.primary_motivation},
       {"field": "typical tasks", "value": persona.typical_tasks}])
show_json(persona, "Persona (full)")

md(f"### Product `{product.product_id}` v{product.product_version}")
table([{"supported": product.supported, "not_supported_yet": product.not_supported_yet,
        "known_gaps": product.known_gaps}])
show_json(product, "Product (full)")

md(f"### Catalog `{catalog.catalog_id}` v{catalog.catalog_version} - {len(catalog.tables)} tables")
for t in catalog.tables:
    md(f"**{t.name}** - {t.description or '_no description_'} (grain: {t.grain or '-'}, PK: {t.primary_key or '-'})")
    table([{"column": c.name, "type": c.data_type, "nullable": c.nullable,
            "description": c.description, "allowed_values": c.allowed_values} for c in t.columns])
md("**Catalog relationships**")
table([{"from": f"{r.source_table}.{r.source_column}", "to": f"{r.target_table}.{r.target_column}",
        "cardinality": r.cardinality, "verification": r.verification} for r in catalog.relationships])
md(f"Schema-leak identifiers checked in questions: `{', '.join(catalog_identifiers(catalog))}`")

### Persona `finance_controller` v1.0

| field | value |
|---|---|
| role | Finance controller responsible for corporate card spend oversight |
| expertise (domain / sql) | high / low |
| style | concise, business-focused |
| traits | asks in business language, never in table or column names, expects totals and rankings to be in a single reporting currency, follows up when numbers look unusual |
| motivation | Keep corporate card spend within budget and catch anomalies early. |
| typical tasks | Review total card spend by department for a period, Identify top merchants or merchant categories by spend, Spot unusual spending patterns, Compare spend across currencies or periods |

### Product `sample_nl2sql` v0.1.0

| supported | not_supported_yet | known_gaps |
|---|---|---|
| single-table aggregation, filtering by date range, ranking and top-N, joins over documented relationships | currency conversion, forecasting | may ignore currency_code when summing amounts across currencies, department attribution is unreliable (employee-to-department link not verified) |

### Catalog `sample_card_spend` v1.0 - 4 tables

...

**merchants** - _no description_ (grain: -, PK: ['merchant_id'])

| column | type | nullable | description | allowed_values |
|---|---|---|---|---|
| merchant_id | STRING | - | - | - |
| merchant_name | STRING | - | - | - |
| country_code | STRING | - | - | - |

**Catalog relationships**

| from | to | cardinality | verification |
|---|---|---|---|
| card_transactions.employee_id | employees.employee_id | many_to_one | verified |
| employees.department_id | departments.department_id | unknown | unknown |

Schema-leak identifiers checked in questions: `card_transactions, cost_center, country_code, currency_code, department_id, department_name, employee_id, full_name, merchant_category, merchant_id, merchant_name, transaction_date, transaction_id`

---
## 1. Node 1 - Reference Index

**Input:** the catalog above. **LLM task:** describe each table (purpose, grain, metrics,
dimensions, time columns, relationships). **Code then:** rejects anything not in the catalog
(retrying with the errors), downgrades any "verified" relationship the catalog does not verify,
and sets ids/versions/timestamps itself.

In [6]:
rec.stage = "node1"
index = await run_node(lambda: ReferenceIndexGenerator(rec, settings, log).generate(catalog))
store.write_model("reference_index.json", index)
rec.show("node1")

**1 LLM call(s)** in `node1`, total 4.8s

#### Call 1 (json, 4.8s)

In [7]:
md(f"### Output: reference index `{index.index_id}` ({index.llm_provider}/{index.llm_model})")
table([{"table": t.name, "business_purpose": t.business_purpose, "grain": t.grain,
        "key_entities": t.key_entities, "important_columns": t.important_columns,
        "metrics": [m.name for m in t.metrics], "dimensions": t.dimensions,
        "time_columns": t.time_columns} for t in index.tables])
md("**Relationships in the index** (verification after code enforcement)")
table([{"from": f"{t.name}.{r.source_column}", "to": f"{r.target_table}.{r.target_column}",
        "cardinality": r.cardinality, "verification": r.verification}
       for t in index.tables for r in t.relationships])
md("**Uncertainties** (LLM notes + code downgrades)")
md("\n".join(f"- {u}" for u in index.uncertainties) or "_(none)_")
errors = check_index_against_catalog(index, catalog)
md(f"**Integrity re-check against catalog:** {'OK - no errors' if not errors else errors}")
show_json(index, "Reference index (full JSON)")

### Output: reference index `IDX_20261014_102441_7642` (gemini/gemini-3.5-flash-lite)

| table | business_purpose | grain | key_entities | important_columns | metrics | dimensions | time_columns |
|---|---|---|---|---|---|---|---|
| card_transactions | One row per corporate card transaction. | one row per transaction | transaction_id, employee_id, merchant_id | transaction_id, employee_id, merchant_id, transaction_date, amount, currency_code, merchant_category | amount | currency_code, merchant_category | transaction_date |
| employees | One row per employee. | one row per employee | employee_id, department_id | employee_id, full_name, department_id | - | full_name | - |
| departments | One row per department. | one row per department | department_id | department_id, department_name, cost_center | - | department_name, cost_center | - |
| merchants | unknown | unknown | merchant_id | merchant_id, merchant_name, country_code | - | merchant_name, country_code | - |

**Relationships in the index** (verification after code enforcement)

| from | to | cardinality | verification |
|---|---|---|---|
| card_transactions.employee_id | employees.employee_id | many_to_one | verified |
| employees.department_id | departments.department_id | unknown | unknown |

**Uncertainties** (LLM notes + code downgrades)

- Table merchants is missing a business purpose and grain.
- Table merchants columns (merchant_id, merchant_name, country_code) are missing descriptions.

**Integrity re-check against catalog:** OK - no errors

---
## 2. Node 2 - Scenarios + cheat sheets

**Input:** persona, product, reference index, config. **LLM task:** business goals plus a
plausible analysis path (cheat sheet) in one call. **Code then:** checks every scenario on its
own (tables/columns must be in the index, multi-table / unsupported settings, duplicate
signatures), copies join verification from the index (adding a note when it changes something),
tops up if short (never pads), and assigns `SCN_` ids.

In [8]:
rec.stage = "node2"
scenario_result = await run_node(lambda: ScenarioGenerator(rec, settings, log, id_generator=ids).generate(
    persona, product, index, count=SCENARIO_COUNT
))
scenarios = scenario_result.accepted
store.write_models("scenarios.json", scenarios, Scenario)
store.write_models("scenarios_rejected.json",
                   [*scenario_result.rejected, *scenario_result.duplicates], RejectedScenario)
rec.show("node2")

**1 LLM call(s)** in `node2`, total 2.6s

#### Call 1 (json, 2.6s)

In [9]:
md(f"### Output: {len(scenarios)} accepted, {len(scenario_result.rejected)} rejected, "
   f"{len(scenario_result.duplicates)} duplicates")
table([{"id": s.id, "title": s.title, "task_type": s.task_type, "capability": s.capability,
        "tables": s.cheat_sheet.tables} for s in scenarios])
for s in scenarios:
    cs = s.cheat_sheet
    md(f"#### {s.id} - {s.title}\n**Business goal:** {s.business_goal}\n\n"
       f"**Capability:** `{s.capability}` - {s.capability_rationale}\n\n"
       f"**Signature:** `{s.signature}`")
    table([{"part": "tables", "value": cs.tables}, {"part": "columns", "value": cs.columns},
           {"part": "metrics", "value": [f"{m.name}: {m.definition}" for m in cs.metrics]},
           {"part": "dimensions", "value": cs.dimensions}, {"part": "operations", "value": cs.operations},
           {"part": "optional_filters", "value": cs.optional_filters}])
    md("Joins (verification copied from the index):")
    table([{"source": r.source, "target": r.target, "verification": r.verification} for r in cs.relationships])
    md("Semantic considerations (code-added notes start with 'join ...'):\n"
       + ("\n".join(f"- {c}" for c in cs.semantic_considerations) or "_(none)_"))
    md(f"Question guidance: `{s.question_guidance.model_dump(mode='json')}`")
md("### Rejected / duplicate scenarios (with reasons)")
for label, items in (("rejected", scenario_result.rejected), ("duplicate", scenario_result.duplicates)):
    for r in items:
        md(f"- **{label}:** _{r.payload.get('title', '(no title)')}_ -> " + "; ".join(r.reasons))
        show_json(r.payload, "payload the LLM returned")
assert scenarios, "No scenarios accepted - check the rejection reasons above."

### Output: 2 accepted, 0 rejected, 0 duplicates

| id | title | task_type | capability | tables |
|---|---|---|---|---|
| SCN_001 | Top Merchants by Spend | ranking | supported | card_transactions, merchants |
| SCN_002 | Spend Across Currencies | comparison | unsupported | card_transactions |

#### SCN_001 - Top Merchants by Spend
**Business goal:** Identify top merchants by total spend to negotiate corporate discounts.

**Capability:** `supported` - The product supports ranking and top-N queries as well as single-table aggregation.

**Signature:** `ranking|corporate discounts identify merchants negotiate spend top total|card_transactions,merchants|total spend|merchants.merchant_name|group by,limit,order by,sum`

| part | value |
|---|---|
| tables | card_transactions, merchants |
| columns | card_transactions.amount, card_transactions.merchant_id, merchants.merchant_id, merchants.merchant_name |
| metrics | Total Spend: Sum of transaction amounts grouped by merchant |
| dimensions | merchants.merchant_name |
| operations | sum, group by, order by, limit |
| optional_filters | - |

Joins (verification copied from the index):

...

Joins (verification copied from the index):

_(none)_

Semantic considerations (code-added notes start with 'join ...'):
- Transactions occur in multiple currencies; summing amounts directly without conversion is risky.

Question guidance: `{'intent': 'Break down card spending totals by currency code.', 'expected_ambiguity': 'medium', 'phrasing_hints': ['What is our total spend broken down by currency?', 'How much are we spending in each currency code?'], 'terms_to_avoid': ['currency_code', 'card_transactions.amount']}`

### Rejected / duplicate scenarios (with reasons)

---
## 3. Snapshot + Node 3 - Questions

**Snapshot (code only):** for each scenario, all catalog columns of its tables + catalog
relationships touching them (original verification); unknown refs go to `missing_references`.
**LLM task:** questions in the persona's voice, no table/column names. **Code then:** retries on
schema leaks (then rejects the leaky ones, never edits text), checks vague/ambiguity rules,
drops near-duplicates, copies the goal from the scenario, and assigns `Q_` ids.

In [10]:
rec.stage = "node3"
question_result = await run_node(lambda: QuestionGenerator(rec, settings, log, id_generator=SequentialIdGenerator()).generate(
    persona, product, scenarios, catalog, count=QUESTIONS_PER_SCENARIO
))
questions = question_result.questions
snapshots = {s.snapshot_id: s for s in question_result.snapshots}
store.write_models("snapshots.json", question_result.snapshots, ReferenceSnapshot)
store.write_models("questions.json", questions, QuestionCase)
store.write_models("questions_rejected.json", question_result.rejected, RejectedQuestion)

md("### Snapshots (deterministic, from the catalog)")
for snap in question_result.snapshots:
    md(f"**{snap.snapshot_id}** - tables: "
       + ", ".join(f"`{t.name}`({len(t.columns)} cols)" for t in snap.tables)
       + f"; relationships: {len(snap.relationships)}; missing_references: {snap.missing_references or 'none'}")
    show_json(snap, f"{snap.snapshot_id} (full)")
rec.show("node3")

### Snapshots (deterministic, from the catalog)

**SNAP_SCN_001** - tables: `card_transactions`(7 cols), `merchants`(3 cols); relationships: 1; missing_references: none

**SNAP_SCN_002** - tables: `card_transactions`(7 cols); relationships: 1; missing_references: none

**2 LLM call(s)** in `node3`, total 5.2s

#### Call 1 (json, 2.3s)

#### Call 2 (json, 2.9s)

In [11]:
md(f"### Output: {len(questions)} questions, {len(question_result.rejected)} rejected")
table([{"id": q.question_id, "scenario": q.scenario_id,
        "style": q.question_characteristics.style,
        "ambiguity": q.question_characteristics.ambiguity_level,
        "clarification": q.expectations.clarification, "utterance": q.utterance,
        "leak_warnings": q.schema_leak_warnings} for q in questions])
for q in questions:
    md(f"#### {q.question_id}: \"{q.utterance}\"\n**Goal (copied from scenario):** {q.intended_goal}\n\n"
       f"**Intent:** `{q.intent.model_dump(mode='json')}`\n\n"
       "**Interpretations:**\n" + "\n".join(f"- {n}" for n in q.interpretation_notes)
       + "\n\n**Acceptable:** " + "; ".join(q.expectations.acceptable_behaviors)
       + "\n\n**Unacceptable:** " + ("; ".join(q.expectations.unacceptable_behaviors) or "-"))
md("### Rejected questions (with reasons)")
table([{"scenario": r.scenario_id, "utterance": r.payload.get("utterance"), "reasons": r.reasons}
       for r in question_result.rejected])
assert questions, "No questions accepted - check the rejection reasons above."

### Output: 4 questions, 0 rejected

| id | scenario | style | ambiguity | clarification | utterance | leak_warnings |
|---|---|---|---|---|---|---|
| Q_001 | SCN_001 | precise | low | not_expected | Could you list our top 10 merchants by total spend so we can identify negotiation opportunities? | - |
| Q_002 | SCN_001 | precise | low | conditional | Show me the top 5 vendors where we accumulated the highest costs this year. | - |
| Q_003 | SCN_002 | precise | medium | conditional | What is our total spend broken down by currency? | - |
| Q_004 | SCN_002 | investigative | medium | not_expected | How much are we spending in each currency code? | - |

#### Q_001: "Could you list our top 10 merchants by total spend so we can identify negotiation opportunities?"
**Goal (copied from scenario):** Identify top merchants by total spend to negotiate corporate discounts.

**Intent:** `{'task_type': 'ranking', 'metrics': ['total spend'], 'dimensions': ['merchant name'], 'filters': [], 'time_range': None}`

**Interpretations:**
- User wants a descending ranking of merchants based on total card expenditure.
- Assumes standard aggregation across all available transaction data.

**Acceptable:** Returns a ranked list of the top 10 merchants by sum of spend.; Aggregates total spend per merchant correctly.

**Unacceptable:** Exposes database table or column names in the response.; Summing amounts across different currencies without a conversion warning or handling.

#### Q_002: "Show me the top 5 vendors where we accumulated the highest costs this year."
**Goal (copied from scenario):** Identify top merchants by total spend to negotiate corporate discounts.

**Intent:** `{'task_type': 'ranking', 'metrics': ['total spend'], 'dimensions': ['merchant name'], 'filters': [], 'time_range': 'this year'}`

**Interpretations:**
- Refers to the current calendar year for transaction filtering.
- Requires limiting the output to the top 5 vendors by spend.

**Acceptable:** Filters transactions to the current calendar year before ranking the top 5 merchants.; Asks for clarification if 'this year' is ambiguous relative to calendar or fiscal year.

**Unacceptable:** Returns all-time data instead of filtering for the current year.; Fails to group by merchant name.

#### Q_003: "What is our total spend broken down by currency?"
**Goal (copied from scenario):** Compare total card spend across different currencies to spot multi-currency exposure.

**Intent:** `{'task_type': 'comparison', 'metrics': ['total spend'], 'dimensions': ['currency'], 'filters': [], 'time_range': None}`

**Interpretations:**
- The user expects a distinct row or group for each currency code with its corresponding total amount.
- Since currency conversion is not supported, the system should either list totals per currency or explicitly warn about multi-currency summation.

**Acceptable:** Show spending totals grouped by currency code without assuming an exchange rate; Warn the user that direct summation across different currencies without conversion may be misleading

**Unacceptable:** Implicitly convert foreign currencies to USD using a hidden exchange rate without disclosure; Sum all amounts together into a single number while ignoring the currency breakdown

#### Q_004: "How much are we spending in each currency code?"
**Goal (copied from scenario):** Compare total card spend across different currencies to spot multi-currency exposure.

**Intent:** `{'task_type': 'aggregation', 'metrics': ['spend amount'], 'dimensions': ['currency code'], 'filters': [], 'time_range': None}`

**Interpretations:**
- Aims to evaluate total corporate card exposure per currency to identify foreign currency risks.
- Assumes the assistant groups the figures by the currency attribute.

**Acceptable:** Aggregate the transaction amounts per currency category; Display the native currency totals clearly to highlight foreign exchange exposure

**Unacceptable:** Combine all currencies into a single blended total without mentioning the lack of currency conversion

### Rejected questions (with reasons)

_(none)_

---
## 4. Node 4 - Product executor (STAND-IN)

The real product executor is not built yet. This stand-in asks the same LLM to act as an NL2SQL
product: given the question and the snapshot schema it returns one SQL `SELECT`, or
`CLARIFY: ...`. Its output becomes the `ExecutionResult` the evaluator judges.
**The SQL is never executed.** Use `SQL_OVERRIDES` to test your own SQL against the evaluator.

In [12]:
STANDIN_SYSTEM = (
    "You are an NL2SQL assistant for BigQuery. Answer with exactly one SQL SELECT statement in a "
    "```sql block. If the question is genuinely ambiguous, reply only 'CLARIFY: <your question>'."
)


def standin_prompt(question: QuestionCase, snapshot: ReferenceSnapshot) -> str:
    schema = {
        "tables": [{"name": t.name, "description": t.description,
                    "columns": [{"name": c.name, "type": c.data_type, "description": c.description}
                                for c in t.columns]} for t in snapshot.tables],
        "relationships": [r.model_dump(mode="json") for r in snapshot.relationships],
    }
    return f"Schema:\n{json.dumps(schema, indent=2)}\n\nQuestion: {question.utterance}"


def to_execution(question: QuestionCase, i: int, text: str | None, error: str | None,
                 started: datetime, seconds: float) -> ExecutionResult:
    base = {"execution_id": f"EXE_{i:03d}", "question_id": question.question_id,
            "product_id": product.product_id, "submitted_question": question.utterance,
            "started_at": started, "metadata": {"latency_ms": round(seconds * 1000)}}
    if error is not None:
        return EXECUTION_RESULT_ADAPTER.validate_python(
            {**base, "outcome": "error", "error_type": "StandInError", "message": error[:500]})
    body = (text or "").strip()
    if body.upper().startswith("CLARIFY:"):
        return EXECUTION_RESULT_ADAPTER.validate_python(
            {**base, "outcome": "clarification", "message": body.split(":", 1)[1].strip()})
    if "```" in body:
        body = body.split("```", 2)[1].removeprefix("sql").strip()
    return EXECUTION_RESULT_ADAPTER.validate_python({**base, "outcome": "sql_generated", "sql": body})


rec.stage = "node4"
executions: list[ExecutionResult] = []
for i, question in enumerate(questions, start=1):
    started, t0 = datetime.now(UTC), time.perf_counter()
    try:
        prompt = standin_prompt(question, snapshots[question.reference_snapshot_id])
        text, error = await run_node(lambda: rec.generate_text(STANDIN_SYSTEM, prompt)), None
    except LLMProviderError as exc:
        text, error = None, str(exc)
    executions.append(to_execution(question, i, text, error, started, time.perf_counter() - t0))
rec.show("node4")

**4 LLM call(s)** in `node4`, total 3.6s

#### Call 1 (text, 0.8s)

#### Call 2 (text, 1.0s)

#### Call 3 (text, 0.9s)

#### Call 4 (text, 0.8s)

In [13]:
# ---- optional: replace any product SQL with your own, then re-run this cell and section 5 ----
SQL_OVERRIDES: dict[str, str] = {
    # "Q_001": "SELECT ... FROM card_transactions ...",
}
for n, ex in enumerate(executions):
    if ex.question_id in SQL_OVERRIDES:
        executions[n] = EXECUTION_RESULT_ADAPTER.validate_python(
            {**ex.model_dump(include={"execution_id", "question_id", "product_id",
                                      "submitted_question", "started_at", "metadata"}),
             "outcome": "sql_generated", "sql": SQL_OVERRIDES[ex.question_id]})
store.write_with("executions.json", executions, TypeAdapter(list[ExecutionResult]))

md("### Output: executions")
table([{"execution": e.execution_id, "question": e.question_id, "outcome": e.outcome,
        "latency_ms": e.metadata.latency_ms,
        "override": "yes" if e.question_id in SQL_OVERRIDES else ""} for e in executions])
for e in executions:
    payload = getattr(e, "sql", None) or getattr(e, "message", None) or getattr(e, "error_type", "")
    show_text(f"{e.execution_id} ({e.outcome}) for {e.question_id}", payload, open_=True)

### Output: executions

| execution | question | outcome | latency_ms | override |
|---|---|---|---|---|
| EXE_001 | Q_001 | clarification | 819.0 | - |
| EXE_002 | Q_002 | clarification | 1047.0 | - |
| EXE_003 | Q_003 | sql_generated | 911.0 | - |
| EXE_004 | Q_004 | sql_generated | 813.0 | - |

---
## 5. Node 5 - Evaluator

Per execution:
- `sql_generated` -> **deterministic SQL rules** first (parse, SELECT-only, unknown columns,
  joins vs verified relationships, tables not in catalog), then the **LLM review** (skipped when
  the rules already decided, e.g. a syntax error).
- `clarification` -> clarification judge (relevance / necessity -> verdict by code).
- `denied` / `error` / `timeout` -> recorded only.

The **final verdict is computed by code** (rules override the LLM); the LLM's own opinion is in
`llm_verdict`. Bug candidates come only from failed evaluations with supported, >= medium findings.

In [14]:
md("### Deterministic rule pre-check (what the code sees before the LLM)")
rows = []
for e in executions:
    if e.outcome == "sql_generated":
        r = apply_sql_rules(e.sql, settings.sql.dialect, catalog)
        rows.append({"execution": e.execution_id, "parse_error": r.parse_error,
                     "non_select": r.non_select, "tables_not_in_catalog": r.unverifiable_tables,
                     "rule_findings": [f"{f.category}: {f.description}" for f in r.findings],
                     "LLM called?": "no" if r.blocks_llm else "yes"})
table(rows)

rec.stage = "node5"
scenario_by_id = {s.id: s for s in scenarios}
question_by_id = {q.question_id: q for q in questions}
cases = [EvaluationCase(question_by_id[e.question_id],
                        scenario_by_id[question_by_id[e.question_id].scenario_id],
                        snapshots[question_by_id[e.question_id].reference_snapshot_id], e)
         for e in executions]
evaluations = await run_node(lambda: SqlEvaluator(rec, settings, log, catalog=catalog, id_generator=SequentialIdGenerator()).evaluate_many(cases, product))
bugs = [b for ev in evaluations for b in ev.bug_candidates]
store.write_models("evaluations.json", evaluations, EvaluationResult)
store.write_models("bug_candidates.json", bugs, BugCandidate)
rec.show("node5")

### Deterministic rule pre-check (what the code sees before the LLM)

| execution | parse_error | non_select | tables_not_in_catalog | rule_findings | LLM called? |
|---|---|---|---|---|---|
| EXE_003 | - | - | - | - | yes |
| EXE_004 | - | - | - | - | yes |

**4 LLM call(s)** in `node5`, total 5.3s

#### Call 1 (json, 1.0s)

#### Call 2 (json, 1.2s)

#### Call 3 (json, 1.5s)

#### Call 4 (json, 1.6s)

In [15]:
for ev in evaluations:
    q = question_by_id[ev.question_id]
    md(f"### {ev.evaluation_id} - {ev.question_id} ({ev.product_outcome})\n> {q.utterance}\n\n"
       f"**Final verdict: `{ev.verdict}`** | LLM verdict: `{ev.llm_verdict}` | status: `{ev.evaluation_status}`")
    if ev.checks:
        table([{"check": k, "status": v} for k, v in ev.checks.model_dump(mode="json").items()])
    if ev.clarification_evaluation:
        md(f"Clarification judge: `{ev.clarification_evaluation.model_dump(mode='json')}`")
    if ev.findings:
        table([{"id": f.finding_id, "source": f.source, "category": f.category, "severity": f.severity,
                "evidence": f.evidence_status, "known_gap": f.matches_known_gap,
                "description": f.description, "sql_evidence": f.sql_evidence} for f in ev.findings])
    for b in ev.bug_candidates:
        md(f"**Bug candidate {b.bug_id}** ({b.status}): {b.title}  \n"
           f"chain: {b.question_id} -> {b.scenario_id} -> {b.execution_id} -> {b.finding_ids}")
    if ev.notes:
        md("Notes: " + "; ".join(ev.notes))
    if ev.cheat_sheet_conflicts:
        md("Cheat-sheet conflicts: " + "; ".join(ev.cheat_sheet_conflicts))

### EVAL_001 - Q_001 (clarification)
> Could you list our top 10 merchants by total spend so we can identify negotiation opportunities?

**Final verdict: `pass`** | LLM verdict: `None` | status: `evaluated`

Clarification judge: `{'relevance': 'relevant', 'necessity': 'necessary', 'asks_for_given_info': False, 'reason': 'The schema specifies that amounts are in different currencies and must not be summed directly, making clarification on currency handling or filtering necessary.'}`

### EVAL_002 - Q_002 (clarification)
> Show me the top 5 vendors where we accumulated the highest costs this year.

**Final verdict: `pass`** | LLM verdict: `None` | status: `evaluated`

Clarification judge: `{'relevance': 'relevant', 'necessity': 'necessary', 'asks_for_given_info': False, 'reason': 'The schema metadata explicitly states that amounts are in different currencies which must not be summed directly, and the system lacks currency conversion capabilities.'}`

### EVAL_003 - Q_003 (sql_generated)
> What is our total spend broken down by currency?

**Final verdict: `pass`** | LLM verdict: `pass` | status: `evaluated`

| check | status |
|---|---|
| intent_alignment | pass |
| table_selection | pass |
| column_selection | pass |
| join_correctness | not_applicable |
| filter_correctness | not_applicable |
| aggregation_correctness | pass |
| sql_syntax | pass |
| semantic_considerations | pass |

### EVAL_004 - Q_004 (sql_generated)
> How much are we spending in each currency code?

**Final verdict: `pass`** | LLM verdict: `pass` | status: `evaluated`

| check | status |
|---|---|
| intent_alignment | pass |
| table_selection | pass |
| column_selection | pass |
| join_correctness | not_applicable |
| filter_correctness | not_applicable |
| aggregation_correctness | pass |
| sql_syntax | pass |
| semantic_considerations | pass |

---
## 6. Summary

In [16]:
table([{"question": q.question_id, "utterance": q.utterance,
        "outcome": next(e.outcome for e in executions if e.question_id == q.question_id),
        "final_verdict": ev.verdict, "llm_verdict": ev.llm_verdict, "findings": len(ev.findings),
        "bugs": [b.bug_id for b in ev.bug_candidates]}
       for ev in evaluations for q in [question_by_id[ev.question_id]]])

md("**LLM usage per node**")
table([{"node": stage, "calls": len(cs),
        "retries": sum(1 for c in cs if "previous output invalid:" in c.user),
        "errors": sum(1 for c in cs if c.error), "seconds": round(sum(c.seconds for c in cs), 1)}
       for stage in ("node1", "node2", "node3", "node4", "node5")
       for cs in [[c for c in rec.calls if c.stage == stage]]])

(NB_DIR / "llm_calls.json").write_text(json.dumps([asdict(c) for c in rec.calls], indent=2), encoding="utf-8")
md(f"All artifacts and every LLM call (prompts + raw outputs) saved in `{NB_DIR.relative_to(REPO)}`:")
md("\n".join(f"- `{p.name}`" for p in sorted(NB_DIR.iterdir())))

| question | utterance | outcome | final_verdict | llm_verdict | findings | bugs |
|---|---|---|---|---|---|---|
| Q_001 | Could you list our top 10 merchants by total spend so we can identify negotiation opportunities? | clarification | pass | - | 0 | - |
| Q_002 | Show me the top 5 vendors where we accumulated the highest costs this year. | clarification | pass | - | 0 | - |
| Q_003 | What is our total spend broken down by currency? | sql_generated | pass | pass | 0 | - |
| Q_004 | How much are we spending in each currency code? | sql_generated | pass | pass | 0 | - |

**LLM usage per node**

| node | calls | retries | errors | seconds |
|---|---|---|---|---|
| node1 | 1 | 0 | 0 | 4.8 |
| node2 | 1 | 0 | 0 | 2.6 |
| node3 | 2 | 0 | 0 | 5.2 |
| node4 | 4 | 0 | 0 | 3.6 |
| node5 | 4 | 0 | 0 | 5.3 |

All artifacts and every LLM call (prompts + raw outputs) saved in `artifacts\notebook_runs\RUN_20261014_102436_05b4`:

- `bug_candidates.json`
- `evaluations.json`
- `executions.json`
- `llm_calls.json`
- `questions.json`
- `questions_rejected.json`
- `reference_index.json`
- `scenarios.json`
- `scenarios_rejected.json`
- `snapshots.json`

---
## 7. Where to rectify what you observe

| Symptom | Where to change it |
|---|---|
| Index misdescribes a table / invents things | `src/synthetic_user/prompts/reference_index.py`; checks in `references/integrity.py` |
| Scenarios dull, repetitive, or rejected | `prompts/scenario.py`; `config/app.yaml` -> `generation.scenario.*`; checks in `references/cheat_sheet.py`, `nodes/scenario_generator.py` |
| Questions sound unlike the persona / leak names | `prompts/question.py`; persona in `data/personas/*.yaml`; `generation.question.*`; leak rule in `references/schema_leak.py` |
| Evaluator too lenient / strict | `prompts/sql_evaluation.py`, `prompts/clarification.py`; overrides in `nodes/sql_rules.py`, `nodes/sql_evaluator.py` |
| Retries exhausted | `llm.max_validation_retries`; the retry feedback text is shown above under each RETRY call |
| Gemini timeouts / 503 | `LLM_TIMEOUT_SECONDS` above or `llm.timeout_seconds` in config; just re-run |

Every non-obvious rule is recorded in `docs/DECISIONS.md` (D001...).